# 第4章 · 合并与长宽表转换

本单元可以单独打开并从头运行，不依赖其他 Notebook 的变量或输出文件。全部小数据为本课程自编合成数据，不代表真实学生、订单或股票行情。

**学习方法**：先说明每行含义与预期结果，再运行代码；练习答案位于折叠单元格。使用课程 `.venv`，无需下载数据或额外安装依赖。

**阅读参考**：[Python for Data Analysis 对应章节](https://wesmckinney.com/book/data-wrangling)。本单元文字、数据与练习为课程自行编写。

### 1. 两张表的粒度与键

合成订单表每行一笔订单，商品表每行一种商品。同一商品可以出现在多笔订单中，因此合并关系是多对一。

In [1]:
import pandas as pd
orders = pd.DataFrame({
    "订单": [1, 2, 3, 4], "商品": ["T", "C", "T", "X"],
    "数量": [2, 1, 3, 1]})
products = pd.DataFrame({
    "商品": ["T", "C", "J"],
    "名称": ["茶", "咖啡", "果汁"], "单价": [10, 20, 15]})
print(orders)
print(products)

   订单 商品  数量
0   1  T   2
1   2  C   1
2   3  T   3
3   4  X   1
  商品  名称  单价
0  T   茶  10
1  C  咖啡  20
2  J  果汁  15


### 2. 左连接：保留每笔订单

on 明确匹配键；left 保留左表订单。indicator 标记未匹配行，validate 检查商品表的键唯一，避免合并后订单被复制。

In [2]:
joined = orders.merge(products, on="商品", how="left",
    validate="many_to_one", indicator=True)
print(joined)
print(joined["_merge"].value_counts())
assert len(joined) == len(orders)

   订单 商品  数量   名称    单价     _merge
0   1  T   2    茶  10.0       both
1   2  C   1   咖啡  20.0       both
2   3  T   3    茶  10.0       both
3   4  X   1  NaN   NaN  left_only
_merge
both          3
left_only     1
right_only    0
Name: count, dtype: int64


### 3. 内连接与外连接：观察谁消失、谁新增

inner 只保留双方存在的商品匹配；outer 还会显示没有订单的商品。未匹配不代表金额为零，应单独列出。

In [3]:
inner = orders.merge(products, on="商品", how="inner")
outer = orders.merge(products, on="商品", how="outer",
                     indicator=True)
print(len(inner), len(outer))  # 3, 5
print(outer[["订单", "商品", "_merge"]])

3 5
    订单 商品      _merge
0  2.0  C        both
1  NaN  J  right_only
2  1.0  T        both
3  3.0  T        both
4  4.0  X   left_only


**先动手**：哪笔订单在 inner 中消失？哪个商品只在右表中？如果要核对全部订单，该选哪种连接？

In [4]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
print(joined.loc[joined["_merge"] == "left_only", "订单"])
print(outer.loc[outer["_merge"] == "right_only", "商品"])
# 订单4；商品J；以订单为主体时使用left。
```
</details>

### 4. 重复键：合并可能让行数变多

在商品表中故意重复 T，观察两笔 T 订单各匹配两次。解决方法是核实键与版本，而不是合并后随意去重。

In [5]:
bad_products = pd.concat([products, products.iloc[[0]]],
                          ignore_index=True)
expanded = orders.merge(bad_products, on="商品", how="left")
print(len(orders), len(expanded))  # 4, 6
print(bad_products.duplicated("商品").sum())
# 加 validate="many_to_one" 将直接报错，课堂可自行尝试。

4 6
1


### 5. concat：把同结构记录上下接起来

concat 不按商品键寻找匹配。ignore_index 重建行号，不检查业务编号是否重复；拼接后仍要检查订单号。

In [6]:
next_orders = pd.DataFrame({
    "订单": [5, 6], "商品": ["C", "T"], "数量": [2, 1]})
all_orders = pd.concat([orders, next_orders], ignore_index=True)
print(all_orders)
print(all_orders["订单"].is_unique)

   订单 商品  数量
0   1  T   2
1   2  C   1
2   3  T   3
3   4  X   1
4   5  C   2
5   6  T   1
True


### 6. concat 的另一方向：横向按索引对齐

axis=1 横向拼接时按索引标签对齐，不会因为行数相同就按位置贴在一起。先检查索引含义。

In [7]:
left = pd.Series([80, 90], index=["001", "002"], name="数学")
right = pd.Series([70, 85], index=["002", "001"], name="英语")
print(pd.concat([left, right], axis=1))

     数学  英语
001  80  85
002  90  70


### 7. 长表转宽表：pivot 不做统计

长表每行是一名学生的一门课成绩；宽表每行一名学生、每列一门课。pivot 要求“学号—课程”组合唯一。

In [8]:
long_scores = pd.DataFrame({
    "学号": ["001", "001", "002", "002"],
    "课程": ["数学", "英语", "数学", "英语"],
    "成绩": [80, 90, 70, 85]})
wide = long_scores.pivot(index="学号", columns="课程", values="成绩")
print(wide)

课程   数学  英语
学号         
001  80  90
002  70  85


### 8. 宽表转长表：melt 保留身份列

id_vars 指定不被展开的标识列。melt 将列名变成课程字段，将原来的单元格变成成绩字段。

In [9]:
back = wide.reset_index().melt(id_vars="学号",
    var_name="课程", value_name="成绩")
print(back.sort_values(["学号", "课程"]))
assert len(back) == len(long_scores)

    学号  课程  成绩
0  001  数学  80
2  001  英语  90
1  002  数学  70
3  002  英语  85


### 9. 重复组合：先决定是否需要聚合

加入一次补考后，同一学生同一课程出现两次。pivot_table 可以聚合；取最大值代表“保留最高成绩”，这是业务规则。

In [10]:
retake = pd.DataFrame({"学号": ["002"],
                       "课程": ["数学"], "成绩": [88]})
attempts = pd.concat([long_scores, retake], ignore_index=True)
best = attempts.pivot_table(index="学号", columns="课程",
    values="成绩", aggfunc="max")
print(best)

课程   数学  英语
学号         
001  80  90
002  88  85


**先动手**：将 max 改为 mean，002 的数学成绩变成多少？两种答案分别回答什么问题？

In [11]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
average = attempts.pivot_table(index="学号", columns="课程",
    values="成绩", aggfunc="mean")
print(average.loc["002", "数学"])  # 79：两次考试的均分，而非最高分。
```
</details>

### 10. 多层索引：把组合键显式写出来

MultiIndex 可以表示“学号—课程”这类组合标签。先学 set_index 和 reset_index，再选学 stack、unstack。

In [12]:
indexed = long_scores.set_index(["学号", "课程"]).sort_index()
print(indexed.loc[("001", "数学"), "成绩"])
print(indexed.unstack("课程"))
print(indexed.reset_index())

80
     成绩    
课程   数学  英语
学号         
001  80  90
002  70  85
    学号  课程  成绩
0  001  数学  80
1  001  英语  90
2  002  数学  70
3  002  英语  85


### 11. 综合检查：只汇总匹配成功的金额

价格未知的 X 订单不能当作零金额。先列出未匹配订单，再汇总可计算的部分，并说明覆盖范围。

In [13]:
matched = joined.loc[joined["_merge"] == "both"].copy()
matched["金额"] = matched["数量"] * matched["单价"]
print(matched[["订单", "名称", "金额"]])
print("已知金额合计：", matched["金额"].sum())  # 70
print("未匹配订单数：", (joined["_merge"] == "left_only").sum())

   订单  名称    金额
0   1   茶  20.0
1   2  咖啡  20.0
2   3   茶  30.0
已知金额合计： 70.0
未匹配订单数： 1


**先动手**：将商品 X 补录为“水”、单价 5，重新合并全部原订单。总金额应是多少？

In [14]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
fixed = pd.concat([products, pd.DataFrame({
    "商品": ["X"], "名称": ["水"], "单价": [5]})], ignore_index=True)
complete = orders.merge(fixed, on="商品", how="left", validate="many_to_one")
print((complete["数量"] * complete["单价"]).sum())  # 75
```
</details>